# 🧠 FinScope AI — AI Service Notebook
> Connect to your live Supabase data, run ML models, generate Claude AI insights, and build the backend brain of the app.

---
**Sections:**
1. 🔌 Connect to Supabase
2. 📊 Load & Explore Transaction Data
3. 🧹 Data Cleaning & Feature Engineering
4. 📈 Spending Analytics & Visualizations
5. 🤖 ML: Spending Anomaly Detection
6. 🔮 ML: Next-Month Spending Forecast
7. 🏷️ ML: Auto Category Classifier
8. 🧬 Claude AI Financial Insights Generator
9. 💾 Write AI Insights Back to Supabase
10. 🚀 Export Model for Production


## 🔌 1. Setup & Connect to Supabase

In [1]:

import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from datetime import datetime, timedelta
from collections import defaultdict

# Supabase client
from supabase import create_client, Client

# Scikit-learn
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

print('✅ All imports successful')

✅ All imports successful


In [28]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env file

SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print('✅ Connected to Supabase:', SUPABASE_URL)

SupabaseException: supabase_url is required

## 📊 2. Load & Explore Transaction Data

In [7]:
# ── Load all transactions from Supabase ──────────────────────────────────────
# Fetches ALL users' transactions (for global ML model training)
# To filter by user: .eq('user_id', USER_ID)

def load_transactions(user_id=None, limit=1000):
    """Load transactions from Supabase into a DataFrame."""
    query = supabase.table('user_transactions').select('*').limit(limit)
    if user_id:
        query = query.eq('user_id', user_id)
    result = query.order('created_at', desc=True).execute()
    
    if not result.data:
        print('⚠️  No data found. Make sure you have transactions in the app.')
        return pd.DataFrame()
    
    df = pd.DataFrame(result.data)
    df['created_at'] = pd.to_datetime(df['created_at'])
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
    df['month'] = df['created_at'].dt.to_period('M')
    df['day_of_week'] = df['created_at'].dt.day_name()
    df['hour'] = df['created_at'].dt.hour
    df['week'] = df['created_at'].dt.isocalendar().week
    df['signed_amount'] = df.apply(
        lambda r: r['amount'] if r['type'] == 'deposit' else -r['amount'], axis=1
    )
    return df

df = load_transactions()

if not df.empty:
    print(f'📦 Loaded {len(df)} transactions')
    print(f'👥 Unique users: {df["user_id"].nunique()}')
    print(f'📅 Date range: {df["created_at"].min().date()} → {df["created_at"].max().date()}')
    print(f'\n🗂️  Columns: {list(df.columns)}')
    display(df.head(10))

⚠️  No data found. Make sure you have transactions in the app.


In [9]:
# ── Load user finances summary ────────────────────────────────────────────────

def load_user_finances():
    result = supabase.table('user_finances').select('*').execute()
    if not result.data:
        return pd.DataFrame()
    finances = pd.DataFrame(result.data)
    finances['balance'] = pd.to_numeric(finances['balance'], errors='coerce')
    finances['total_income'] = pd.to_numeric(finances['total_income'], errors='coerce')
    finances['total_expenses'] = pd.to_numeric(finances['total_expenses'], errors='coerce')
    finances['savings_rate'] = (
        (finances['total_income'] - finances['total_expenses']) / 
        finances['total_income'].replace(0, np.nan)
    ).fillna(0) * 100
    return finances

finances_df = load_user_finances()
if not finances_df.empty:
    print(f'💰 User Finances Summary ({len(finances_df)} users)')
    display(finances_df[['user_id', 'balance', 'total_income', 'total_expenses', 'savings_rate']].head())

In [10]:
# ── Basic stats ───────────────────────────────────────────────────────────────
if not df.empty:
    expenses = df[df['type'] == 'expense']
    deposits = df[df['type'] == 'deposit']
    
    print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    print('📊 FinScope AI — Dataset Overview')
    print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    print(f'  Total transactions : {len(df):,}')
    print(f'  Expenses           : {len(expenses):,}  (${expenses["amount"].sum():,.2f} total)')
    print(f'  Deposits           : {len(deposits):,}  (${deposits["amount"].sum():,.2f} total)')
    print(f'  Avg expense        : ${expenses["amount"].mean():.2f}')
    print(f'  Largest expense    : ${expenses["amount"].max():.2f}')
    print(f'  Categories         : {expenses["category"].nunique()} unique')
    print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    print()
    print('🏷️  Spending by Category:')
    cat_summary = expenses.groupby('category')['amount'].agg(['sum', 'count', 'mean'])
    cat_summary.columns = ['Total ($)', 'Count', 'Avg ($)']
    cat_summary = cat_summary.sort_values('Total ($)', ascending=False)
    cat_summary['Total ($)'] = cat_summary['Total ($)'].map('${:,.2f}'.format)
    cat_summary['Avg ($)'] = cat_summary['Avg ($)'].map('${:.2f}'.format)
    display(cat_summary)

## 📈 3. Visualizations

In [11]:
# ── Plot 1: Spending by Category (Horizontal Bar) ─────────────────────────────
if not df.empty and len(expenses) > 0:
    COLORS = {
        'Groceries': '#3b82f6', 'Subscriptions': '#8b5cf6', 'Shopping': '#a78bfa',
        'Transport': '#06b6d4', 'Dining': '#f59e0b', 'Bills': '#ef4444',
        'Health': '#10b981', 'Entertainment': '#ec4899', 'Education': '#f97316', 'Other': '#6b7280'
    }

    cat_data = expenses.groupby('category')['amount'].sum().sort_values(ascending=True)
    colors = [COLORS.get(c, '#6b7280') for c in cat_data.index]

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor('#0f1729')
    ax.set_facecolor('#0f1729')

    bars = ax.barh(cat_data.index, cat_data.values, color=colors, height=0.6, edgecolor='none')

    for bar, val in zip(bars, cat_data.values):
        ax.text(bar.get_width() + max(cat_data.values) * 0.01, bar.get_y() + bar.get_height() / 2,
                f'${val:,.0f}', va='center', ha='left', color='white', fontsize=9, fontweight='bold')

    ax.set_xlabel('Total Spent ($)', color='#94a3b8', fontsize=10)
    ax.set_title('Spending by Category', color='white', fontsize=14, fontweight='bold', pad=16)
    ax.tick_params(colors='#94a3b8', labelsize=9)
    ax.spines[:].set_color('#1e2a4a')
    ax.xaxis.label.set_color('#94a3b8')
    plt.tight_layout()
    plt.savefig('chart_categories.png', dpi=150, bbox_inches='tight', facecolor='#0f1729')
    plt.show()
    print('✅ Saved chart_categories.png')

In [12]:
# ── Plot 2: Monthly Spending Trend ────────────────────────────────────────────
if not df.empty and len(expenses) > 0:
    monthly = expenses.groupby('month')['amount'].sum().reset_index()
    monthly['month_str'] = monthly['month'].astype(str)

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor('#0f1729')
    ax.set_facecolor('#0f1729')

    x = range(len(monthly))
    ax.fill_between(x, monthly['amount'], alpha=0.25, color='#3b82f6')
    ax.plot(x, monthly['amount'], color='#3b82f6', linewidth=2.5, marker='o', markersize=5)

    for xi, row in zip(x, monthly.itertuples()):
        ax.annotate(f'${row.amount:,.0f}', (xi, row.amount), textcoords='offset points',
                    xytext=(0, 10), ha='center', color='white', fontsize=8)

    ax.set_xticks(list(x))
    ax.set_xticklabels(monthly['month_str'], rotation=30, ha='right', color='#94a3b8', fontsize=9)
    ax.set_ylabel('Total Expenses ($)', color='#94a3b8')
    ax.set_title('Monthly Spending Trend', color='white', fontsize=14, fontweight='bold', pad=16)
    ax.tick_params(colors='#94a3b8')
    ax.spines[:].set_color('#1e2a4a')
    plt.tight_layout()
    plt.savefig('chart_monthly_trend.png', dpi=150, bbox_inches='tight', facecolor='#0f1729')
    plt.show()
    print('✅ Saved chart_monthly_trend.png')

In [13]:
# ── Plot 3: Spending Heatmap (Day of Week × Category) ────────────────────────
if not df.empty and len(expenses) > 10:
    day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    pivot = expenses.pivot_table(
        values='amount', index='day_of_week', columns='category', aggfunc='sum', fill_value=0
    ).reindex(day_order, fill_value=0)

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor('#0f1729')
    ax.set_facecolor('#0f1729')

    sns.heatmap(pivot, cmap='Blues', ax=ax, linewidths=0.5, linecolor='#0f1729',
                cbar_kws={'label': 'Amount ($)'},
                annot=pivot.applymap(lambda v: f'${v:,.0f}' if v > 0 else ''),
                fmt='', annot_kws={'size': 8, 'color': 'white'})

    ax.set_title('Spending Heatmap: Day × Category', color='white', fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('', color='white'); ax.set_ylabel('', color='white')
    ax.tick_params(colors='#94a3b8', labelsize=9)
    plt.tight_layout()
    plt.savefig('chart_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0f1729')
    plt.show()
    print('✅ Saved chart_heatmap.png')

## 🤖 4. Anomaly Detection — Flag Unusual Transactions

In [14]:
# ── Isolation Forest: Detect unusual spend amounts ───────────────────────────
# This flags transactions that are statistically unusual for each user

def detect_anomalies(expenses_df, contamination=0.05):
    """
    Flag transactions that are anomalously large or unusual.
    Returns the dataframe with an `is_anomaly` column.
    """
    if len(expenses_df) < 10:
        print('⚠️  Not enough data for anomaly detection (need ≥10 expenses)')
        return expenses_df

    # Features for anomaly detection
    features = expenses_df[['amount']].copy()
    
    # Also add day-of-week signal (encode numerically)
    day_map = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4,'Saturday':5,'Sunday':6}
    features['day_num'] = expenses_df['day_of_week'].map(day_map).fillna(0)
    
    model = IsolationForest(
        contamination=contamination,  # expect ~5% anomalies
        random_state=42,
        n_estimators=100
    )
    
    preds = model.fit_predict(features)
    scores = model.score_samples(features)  # lower = more anomalous
    
    result = expenses_df.copy()
    result['is_anomaly'] = preds == -1  # -1 means anomaly
    result['anomaly_score'] = scores
    result['anomaly_rank'] = result['anomaly_score'].rank()  # lower rank = more suspicious
    
    return result


if not df.empty and len(expenses) >= 10:
    expenses_flagged = detect_anomalies(expenses)
    anomalies = expenses_flagged[expenses_flagged['is_anomaly']].sort_values('anomaly_score')
    
    print(f'🚨 Anomalies detected: {len(anomalies)} out of {len(expenses)} expense transactions')
    print()
    if len(anomalies) > 0:
        print('⚠️  Most suspicious transactions:')
        display(anomalies[['created_at','name','category','amount','anomaly_score']].head(10))
else:
    print('ℹ️  Add at least 10 expense transactions in the app to run anomaly detection.')

ℹ️  Add at least 10 expense transactions in the app to run anomaly detection.


In [15]:
# ── Plot anomalies on the spending chart ─────────────────────────────────────
if not df.empty and 'is_anomaly' in dir():
    try:
        fig, ax = plt.subplots(figsize=(12, 5))
        fig.patch.set_facecolor('#0f1729')
        ax.set_facecolor('#0f1729')

        normal = expenses_flagged[~expenses_flagged['is_anomaly']]
        anom = expenses_flagged[expenses_flagged['is_anomaly']]

        ax.scatter(normal.index, normal['amount'], color='#3b82f6', alpha=0.6, s=40, label='Normal')
        ax.scatter(anom.index, anom['amount'], color='#ef4444', s=80, zorder=5,
                   edgecolors='white', linewidths=0.8, label='Anomaly 🚨')

        for _, row in anom.iterrows():
            ax.annotate(f"{row['name'][:12]}\n${row['amount']:.0f}",
                        (row.name, row['amount']), textcoords='offset points', xytext=(8, 4),
                        color='#fca5a5', fontsize=7)

        ax.set_ylabel('Amount ($)', color='#94a3b8')
        ax.set_title('Transaction Anomaly Detection', color='white', fontsize=13, fontweight='bold')
        ax.tick_params(colors='#94a3b8', labelbottom=False)
        ax.spines[:].set_color('#1e2a4a')
        ax.legend(facecolor='#1e2a4a', edgecolor='#3b82f6', labelcolor='white')
        plt.tight_layout()
        plt.savefig('chart_anomalies.png', dpi=150, bbox_inches='tight', facecolor='#0f1729')
        plt.show()
    except:
        print('⚠️  Run the anomaly detection cell above first.')

## 🔮 5. Spending Forecast — Predict Next Month

In [16]:
# ── Linear Regression: Predict next-month spending ───────────────────────────
# Works per user or globally; needs at least 3 months of data

def forecast_next_month(expenses_df, user_id=None):
    """
    Forecast next month's spending using linear regression on historical monthly totals.
    Returns predicted amount and confidence interval.
    """
    if user_id:
        data = expenses_df[expenses_df['user_id'] == user_id]
    else:
        data = expenses_df

    monthly = data.groupby('month')['amount'].sum().reset_index()
    monthly = monthly.sort_values('month')

    if len(monthly) < 3:
        return {
            'status': 'insufficient_data',
            'message': f'Need at least 3 months of data (have {len(monthly)})'
        }

    X = np.arange(len(monthly)).reshape(-1, 1)
    y = monthly['amount'].values

    model = LinearRegression()
    model.fit(X, y)

    next_idx = np.array([[len(monthly)]])
    predicted = model.predict(next_idx)[0]
    
    # Simple confidence interval (2x std of residuals)
    residuals = y - model.predict(X)
    margin = 2 * residuals.std()

    # Trend direction
    trend = 'increasing' if model.coef_[0] > 0 else 'decreasing'
    trend_pct = abs(model.coef_[0] / y.mean() * 100) if y.mean() > 0 else 0

    return {
        'status': 'ok',
        'predicted': max(0, predicted),
        'low': max(0, predicted - margin),
        'high': predicted + margin,
        'trend': trend,
        'trend_pct_per_month': round(trend_pct, 1),
        'monthly_history': monthly.to_dict('records'),
        'r2_score': model.score(X, y),
        'months_of_data': len(monthly)
    }


if not df.empty and len(expenses) > 0:
    forecast = forecast_next_month(expenses)

    if forecast['status'] == 'ok':
        print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print('🔮 Next-Month Spending Forecast')
        print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'  Predicted spending : ${forecast["predicted"]:,.2f}')
        print(f'  Confidence range   : ${forecast["low"]:,.2f} — ${forecast["high"]:,.2f}')
        print(f'  Trend              : {forecast["trend"]} +{forecast["trend_pct_per_month"]}%/month')
        print(f'  Model R² score     : {forecast["r2_score"]:.3f}')
        print(f'  Based on           : {forecast["months_of_data"]} months of data')
        print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    else:
        print(f'ℹ️  {forecast["message"]}')

In [17]:
# ── Plot forecast with confidence band ───────────────────────────────────────
if not df.empty and forecast.get('status') == 'ok':
    history = pd.DataFrame(forecast['monthly_history'])
    history['month_str'] = history['month'].astype(str)

    next_month_label = str((
        pd.Period(history['month_str'].iloc[-1]) + 1
    ))

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor('#0f1729')
    ax.set_facecolor('#0f1729')

    x = list(range(len(history)))
    ax.fill_between(x, history['amount'], alpha=0.2, color='#3b82f6')
    ax.plot(x, history['amount'], color='#3b82f6', linewidth=2.5, marker='o', markersize=6, label='Actual')

    # Forecast point
    fx = len(history)
    ax.plot([fx-1, fx], [history['amount'].iloc[-1], forecast['predicted']],
            color='#8b5cf6', linewidth=2, linestyle='--')
    ax.scatter([fx], [forecast['predicted']], color='#8b5cf6', s=100, zorder=5, label='Forecast')
    ax.fill_between([fx-0.1, fx+0.1],
                    [forecast['low']]*2, [forecast['high']]*2,
                    alpha=0.3, color='#8b5cf6')

    ax.annotate(f"${forecast['predicted']:,.0f}\n(forecast)",
                (fx, forecast['predicted']), textcoords='offset points', xytext=(10, 6),
                color='#c4b5fd', fontsize=9, fontweight='bold')

    all_labels = list(history['month_str']) + [next_month_label]
    ax.set_xticks(range(len(all_labels)))
    ax.set_xticklabels(all_labels, rotation=30, ha='right', color='#94a3b8', fontsize=9)
    ax.set_ylabel('Spending ($)', color='#94a3b8')
    ax.set_title('Monthly Spending Forecast', color='white', fontsize=14, fontweight='bold', pad=16)
    ax.tick_params(colors='#94a3b8')
    ax.spines[:].set_color('#1e2a4a')
    ax.legend(facecolor='#1e2a4a', edgecolor='#3b82f6', labelcolor='white', fontsize=9)
    plt.tight_layout()
    plt.savefig('chart_forecast.png', dpi=150, bbox_inches='tight', facecolor='#0f1729')
    plt.show()
    print('✅ Saved chart_forecast.png')

## 🏷️ 6. Auto Category Classifier

In [18]:
# ── Train a classifier to auto-assign categories to new transactions ─────────
# This powers the "smart categorization" feature in the app

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import re

# Rule-based fallback (no training data needed)
CATEGORY_KEYWORDS = {
    'Subscriptions': ['spotify', 'netflix', 'disney', 'youtube', 'apple', 'hulu', 'prime',
                      'membership', 'subscription', 'gym', 'annual', 'monthly plan'],
    'Dining':        ['starbucks', 'coffee', 'restaurant', 'uber eats', 'doordash', 'grubhub',
                      'mcdonalds', 'pizza', 'sushi', 'food', 'meal', 'lunch', 'dinner', 'cafe'],
    'Transport':     ['uber', 'lyft', 'taxi', 'gas', 'petrol', 'parking', 'metro', 'bus',
                      'train', 'transit', 'fuel', 'toll', 'car wash'],
    'Shopping':      ['amazon', 'nike', 'zara', 'h&m', 'target', 'walmart', 'ebay', 'shop',
                      'clothing', 'shoes', 'electronics', 'gadget', 'order'],
    'Bills':         ['electricity', 'internet', 'wifi', 'phone', 'rent', 'mortgage', 'utility',
                      'water', 'insurance', 'council tax', 'bill'],
    'Groceries':     ['tesco', 'sainsbury', 'waitrose', 'aldi', 'lidl', 'grocery', 'supermarket',
                      'whole foods', 'trader joe', 'kroger', 'publix'],
    'Health':        ['pharmacy', 'doctor', 'dentist', 'hospital', 'clinic', 'medicine',
                      'prescription', 'health', 'medical', 'gym'],
    'Entertainment': ['cinema', 'movie', 'theater', 'concert', 'event', 'ticket', 'game',
                      'sport', 'netflix', 'steam', 'playstation', 'xbox'],
    'Education':     ['course', 'udemy', 'coursera', 'school', 'university', 'tuition',
                      'book', 'textbook', 'education', 'class', 'lesson'],
}

def rule_based_category(name: str) -> str:
    """Fast keyword-based fallback categorization."""
    name_lower = name.lower()
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in name_lower for kw in keywords):
            return category
    return 'Other'


def train_ml_classifier(expenses_df):
    """Train a TF-IDF + Random Forest classifier on real transaction names."""
    if len(expenses_df) < 30:
        print('⚠️  Not enough labeled transactions to train ML model (need ≥30). Using rule-based.')
        return None

    X = expenses_df['name'].astype(str).str.lower()
    y = expenses_df['category']

    if y.nunique() < 2:
        print('⚠️  Only 1 category in data — need diversity to train classifier.')
        return None

    clf = Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), max_features=500)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
    ])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    clf.fit(X_train, y_train)

    print('📊 Category Classifier — Test Set Report:')
    print(classification_report(y_test, clf.predict(X_test)))
    return clf


# Train (or fallback)
if not df.empty and len(expenses) >= 30:
    clf_model = train_ml_classifier(expenses)
else:
    clf_model = None
    print('ℹ️  Using rule-based categorizer (add more transactions to enable ML model).')


def predict_category(transaction_name: str) -> dict:
    """Predict category for a new transaction name."""
    if clf_model:
        pred = clf_model.predict([transaction_name.lower()])[0]
        proba = clf_model.predict_proba([transaction_name.lower()])[0]
        confidence = round(float(max(proba)) * 100, 1)
        return {'category': pred, 'confidence': confidence, 'method': 'ml'}
    else:
        cat = rule_based_category(transaction_name)
        return {'category': cat, 'confidence': 85.0, 'method': 'rules'}


# Demo predictions
test_names = [
    'Starbucks Coffee', 'Uber ride home', 'Netflix monthly',
    'Electricity bill', 'Amazon order', 'Doctor appointment',
    'Concert tickets', 'Tesco groceries', 'Udemy course'
]

print('\n🏷️  Category Predictions:')
print('─' * 50)
for name in test_names:
    result = predict_category(name)
    print(f'  {name:<28} → {result["category"]:<16} ({result["confidence"]}% via {result["method"]})')

ℹ️  Using rule-based categorizer (add more transactions to enable ML model).

🏷️  Category Predictions:
──────────────────────────────────────────────────
  Starbucks Coffee             → Dining           (85.0% via rules)
  Uber ride home               → Transport        (85.0% via rules)
  Netflix monthly              → Subscriptions    (85.0% via rules)
  Electricity bill             → Bills            (85.0% via rules)
  Amazon order                 → Shopping         (85.0% via rules)
  Doctor appointment           → Health           (85.0% via rules)
  Concert tickets              → Entertainment    (85.0% via rules)
  Tesco groceries              → Groceries        (85.0% via rules)
  Udemy course                 → Education        (85.0% via rules)


## 🧬 7. Claude AI Financial Insights Generator

In [19]:
# ── Generate personalized financial insights using Claude ─────────────────────
# This is what powers the AI Insights panel in the app dashboard

import anthropic

def build_financial_context(user_id, expenses_df, finances_df):
    """Build a structured financial summary to send to Claude."""
    user_expenses = expenses_df[expenses_df['user_id'] == user_id] if user_id else expenses_df
    user_finance = finances_df[finances_df['user_id'] == user_id].iloc[0] if (
        user_id and not finances_df.empty and (finances_df['user_id'] == user_id).any()
    ) else None

    # Category breakdown
    cat_totals = user_expenses.groupby('category')['amount'].sum().sort_values(ascending=False).to_dict()

    # Monthly trend
    monthly = user_expenses.groupby('month')['amount'].sum().sort_index()
    monthly_list = [(str(m), round(v, 2)) for m, v in monthly.items()]

    # Top transactions
    top_txns = user_expenses.nlargest(5, 'amount')[['name', 'amount', 'category']].to_dict('records')

    # Recurring detection (same name, multiple months)
    recurring = user_expenses.groupby('name').filter(lambda g: g['month'].nunique() >= 2)
    recurring_names = recurring['name'].value_counts().head(5).index.tolist()

    context = {
        'balance': round(float(user_finance['balance']), 2) if user_finance is not None else None,
        'total_income': round(float(user_finance['total_income']), 2) if user_finance is not None else None,
        'total_expenses': round(float(user_finance['total_expenses']), 2) if user_finance is not None else None,
        'savings_rate': round(float(user_finance['savings_rate']), 1) if user_finance is not None else None,
        'category_breakdown': {k: round(v, 2) for k, v in cat_totals.items()},
        'monthly_trend': monthly_list,
        'top_5_expenses': top_txns,
        'recurring_subscriptions': recurring_names,
        'total_transactions': len(user_expenses),
    }
    return context


def generate_ai_insights(context: dict, api_key: str) -> dict:
    """Send financial context to Claude and get structured insights back."""
    if api_key == 'YOUR_ANTHROPIC_API_KEY_HERE':
        print('⚠️  Set your ANTHROPIC_API_KEY above to use Claude AI insights.')
        return demo_insights(context)

    client = anthropic.Anthropic(api_key=api_key)

    prompt = f"""You are FinScope AI, a smart financial advisor inside a personal finance app.

Analyze this user's financial data and return a JSON object with exactly these keys:
- health_score: integer 0-100 (overall financial health)
- health_label: string like "Good", "Fair", "Needs Attention"
- summary: 1 sentence overview of their finances
- insights: array of 3 insight objects, each with: {{"type": "warning|tip|positive", "title": str, "body": str, "icon": emoji}}
- biggest_risk: string describing the biggest financial risk
- top_tip: string with the single most impactful advice
- projected_savings_3m: float estimate of savings in 3 months if they follow your tip

Financial Data:
{json.dumps(context, indent=2)}

Return only valid JSON, no markdown, no extra text."""

    message = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=1024,
        messages=[{'role': 'user', 'content': prompt}]
    )

    raw = message.content[0].text.strip()
    # Strip markdown code fences if present
    raw = re.sub(r'^```json\s*|^```\s*|\s*```$', '', raw, flags=re.MULTILINE).strip()

    return json.loads(raw)


def demo_insights(context: dict) -> dict:
    """Demo insights when no API key is set."""
    total_exp = context.get('total_expenses') or sum(context.get('category_breakdown', {}).values())
    total_inc = context.get('total_income') or 0
    savings_rate = context.get('savings_rate') or (
        round((total_inc - total_exp) / total_inc * 100, 1) if total_inc > 0 else 0
    )

    top_cat = max(context.get('category_breakdown', {}).items(), key=lambda x: x[1], default=('Other', 0))
    score = min(100, max(0, 50 + savings_rate))

    return {
        'health_score': int(score),
        'health_label': 'Good' if score >= 70 else ('Fair' if score >= 40 else 'Needs Attention'),
        'summary': f'You\'ve spent ${total_exp:,.0f} total with a {savings_rate:.0f}% savings rate.',
        'insights': [
            {'type': 'warning', 'title': f'High {top_cat[0]} spend',
             'body': f'${top_cat[1]:,.0f} on {top_cat[0]} is your biggest category.',
             'icon': '⚠️'},
            {'type': 'tip', 'title': 'Set a monthly budget',
             'body': 'Creating category budgets can reduce overspending by 20%.',
             'icon': '💡'},
            {'type': 'positive', 'title': 'Consistent tracking',
             'body': f'You\'ve logged {context.get("total_transactions", 0)} transactions — great habit!',
             'icon': '✅'},
        ],
        'biggest_risk': f'Overspending on {top_cat[0]}',
        'top_tip': 'Review your subscriptions and cancel unused ones.',
        'projected_savings_3m': round(total_exp * 0.15, 2)
    }


# Generate insights for all users (or a specific one)
if not df.empty:
    # Use first user as demo
    demo_user = df['user_id'].iloc[0]
    ctx = build_financial_context(demo_user, expenses, finances_df)
    insights = generate_ai_insights(ctx, ANTHROPIC_API_KEY)

    print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    print(f'🧠 FinScope AI Insights — User {demo_user[:8]}...')
    print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    print(f'  Health Score : {insights["health_score"]}/100 ({insights["health_label"]})')
    print(f'  Summary      : {insights["summary"]}')
    print(f'  Biggest Risk : {insights["biggest_risk"]}')
    print(f'  Top Tip      : {insights["top_tip"]}')
    print(f'  3-Month Save : ${insights.get("projected_savings_3m", 0):,.2f}')
    print()
    print('  Insights:')
    for ins in insights.get('insights', []):
        print(f'    {ins["icon"]} [{ins["type"].upper()}] {ins["title"]}: {ins["body"]}')
    print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
else:
    print('ℹ️  Add transactions in the app first, then re-run this cell.')

ℹ️  Add transactions in the app first, then re-run this cell.


## 💾 8. Write AI Results Back to Supabase

In [21]:
# ── Push AI insights to Supabase so the app can display them ─────────────────
# This requires an `ai_insights` table in Supabase
#
# SQL to create the table (run in Supabase SQL Editor):
#
#   CREATE TABLE IF NOT EXISTS ai_insights (
#     id              uuid DEFAULT gen_random_uuid() PRIMARY KEY,
#     user_id         uuid REFERENCES auth.users(id) ON DELETE CASCADE,
#     health_score    int,
#     health_label    text,
#     summary         text,
#     insights_json   jsonb,
#     biggest_risk    text,
#     top_tip         text,
#     projected_save  numeric,
#     generated_at    timestamptz DEFAULT now(),
#     model_version   text DEFAULT 'v1'
#   );
#   ALTER TABLE ai_insights ENABLE ROW LEVEL SECURITY;
#   CREATE POLICY "users see own insights" ON ai_insights
#     FOR SELECT USING (auth.uid() = user_id);

def push_insights_to_supabase(user_id: str, insights: dict):
    """Write AI insights into the ai_insights Supabase table."""
    payload = {
        'user_id': user_id,
        'health_score': insights.get('health_score'),
        'health_label': insights.get('health_label'),
        'summary': insights.get('summary'),
        'insights_json': json.dumps(insights.get('insights', [])),
        'biggest_risk': insights.get('biggest_risk'),
        'top_tip': insights.get('top_tip'),
        'projected_save': insights.get('projected_savings_3m'),
        'model_version': 'v1'
    }

    # Upsert: update if user already has insights, otherwise insert
    result = supabase.table('ai_insights').upsert(
        payload,
        on_conflict='user_id'
    ).execute()

    if result.data:
        print(f'✅ Insights saved to Supabase for user {user_id[:8]}...')
    else:
        print(f'❌ Failed to save insights: {result}')
    return result


# ── Run for ALL users ─────────────────────────────────────────────────────────
def run_insights_pipeline_all_users():
    """Generate and push AI insights for every user in the DB."""
    if df.empty:
        print('No transaction data available.')
        return

    user_ids = df['user_id'].unique()
    print(f'🚀 Running AI insights pipeline for {len(user_ids)} users...')

    for uid in user_ids:
        try:
            ctx = build_financial_context(uid, expenses, finances_df)
            ins = generate_ai_insights(ctx, ANTHROPIC_API_KEY)
            # Uncomment to actually write to Supabase:
            # push_insights_to_supabase(uid, ins)
            print(f'  ✓ {uid[:8]}... → {ins["health_label"]} ({ins["health_score"]}/100)')
        except Exception as e:
            print(f'  ✗ {uid[:8]}... → Error: {e}')

    print('\nDone! Uncomment push_insights_to_supabase() to write results to Supabase.')


# Preview (doesn't write yet)
if not df.empty:
    run_insights_pipeline_all_users()

## 🚀 9. Full AI Pipeline — One-Click Run

In [22]:
# ── Run everything end-to-end for a single user ───────────────────────────────
def full_pipeline(user_id=None):
    """
    Complete FinScope AI pipeline:
    1. Load data
    2. Anomaly detection
    3. Spending forecast
    4. Category predictions
    5. Claude AI insights
    Returns a complete report dict.
    """
    print('━' * 55)
    print('🚀 FinScope AI — Full Pipeline')
    print('━' * 55)

    # 1. Load
    all_txns = load_transactions(user_id=user_id)
    if all_txns.empty:
        return {'error': 'No transaction data found.'}
    
    exp = all_txns[all_txns['type'] == 'expense']
    print(f'  ✅ Loaded {len(all_txns)} transactions ({len(exp)} expenses)')

    # 2. Anomalies
    anomalies_found = []
    if len(exp) >= 10:
        flagged = detect_anomalies(exp)
        anomalies_found = flagged[flagged['is_anomaly']][['name','amount','category']].to_dict('records')
        print(f'  ✅ Anomaly detection: {len(anomalies_found)} flagged')
    else:
        print('  ⚠️  Skipped anomaly detection (need ≥10 expenses)')

    # 3. Forecast
    fc = forecast_next_month(exp)
    if fc['status'] == 'ok':
        print(f'  ✅ Forecast: ${fc["predicted"]:,.2f} next month ({fc["trend"]})')
    else:
        print(f'  ⚠️  Forecast: {fc["message"]}')

    # 4. Context + Claude Insights
    fin = load_user_finances()
    ctx = build_financial_context(user_id, exp, fin)
    ins = generate_ai_insights(ctx, ANTHROPIC_API_KEY)
    print(f'  ✅ AI Insights: Health {ins["health_score"]}/100 — {ins["health_label"]}')

    # Summary
    report = {
        'user_id': user_id,
        'generated_at': datetime.now().isoformat(),
        'transactions': len(all_txns),
        'anomalies': anomalies_found,
        'forecast': fc,
        'ai_insights': ins,
        'financial_context': ctx
    }

    print()
    print('  📋 Final Report:')
    print(f'     Health Score     : {ins["health_score"]}/100')
    print(f'     Summary          : {ins["summary"]}')
    print(f'     Anomalies Found  : {len(anomalies_found)}')
    if fc['status'] == 'ok':
        print(f'     Next Month Est.  : ${fc["predicted"]:,.2f}')
    print(f'     Top Tip          : {ins["top_tip"]}')
    print('━' * 55)

    return report


# Run the full pipeline!
report = full_pipeline()
print('\n✅ Pipeline complete. Report saved to `report` variable.')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🚀 FinScope AI — Full Pipeline
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚠️  No data found. Make sure you have transactions in the app.

✅ Pipeline complete. Report saved to `report` variable.


In [23]:
# ── Export report to JSON ─────────────────────────────────────────────────────
if 'report' in dir() and isinstance(report, dict) and 'error' not in report:
    # Convert any non-serializable types
    def make_serializable(obj):
        if isinstance(obj, (pd.Period, pd.Timestamp)): return str(obj)
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if isinstance(obj, dict): return {k: make_serializable(v) for k, v in obj.items()}
        if isinstance(obj, list): return [make_serializable(i) for i in obj]
        return obj

    clean_report = make_serializable(report)

    with open('finscope_ai_report.json', 'w') as f:
        json.dump(clean_report, f, indent=2)

    print('✅ Report exported to finscope_ai_report.json')
    print(f'   Size: {len(json.dumps(clean_report)):,} characters')

## 🛠️ 10. Utilities & Helpers

In [24]:
# ── Supabase table inspector ──────────────────────────────────────────────────
def inspect_tables():
    """List and sample all FinScope tables in Supabase."""
    tables = ['user_transactions', 'user_finances', 'user_profiles', 'ai_insights']
    for tbl in tables:
        try:
            res = supabase.table(tbl).select('*').limit(2).execute()
            if res.data:
                cols = list(res.data[0].keys())
                print(f'  ✅ {tbl:<25} → {len(res.data)} sample rows | cols: {cols}')
            else:
                print(f'  📭 {tbl:<25} → empty')
        except Exception as e:
            print(f'  ❌ {tbl:<25} → {str(e)[:60]}')

print('📦 FinScope AI — Supabase Tables:')
inspect_tables()

📦 FinScope AI — Supabase Tables:
  📭 user_transactions         → empty
  📭 user_finances             → empty
  ✅ user_profiles             → 2 sample rows | cols: ['id', 'email', 'full_name', 'created_at', 'updated_at', 'phone', 'date_of_birth', 'gender', 'bio', 'location', 'occupation', 'monthly_income', 'preferred_currency', 'financial_goal', 'avatar_url', 'website', 'twitter', 'linkedin', 'first_name', 'last_name', 'username', 'wallpaper_url', 'card_type', 'card_theme', 'city', 'country', 'marketing_consent']
  ❌ ai_insights               → {'message': "Could not find the table 'public.ai_insights' i


In [25]:
# ── Batch categorize uncategorized transactions ───────────────────────────────
def fix_uncategorized_transactions(dry_run=True):
    """
    Find transactions with category='Other' or null and suggest better categories.
    Set dry_run=False to update them in Supabase.
    """
    if df.empty:
        return

    other_mask = (df['category'].isna()) | (df['category'] == 'Other')
    others = df[other_mask]

    if others.empty:
        print('✅ No uncategorized transactions found!')
        return

    print(f'🔍 Found {len(others)} uncategorized transactions')
    fixes = []

    for _, row in others.iterrows():
        pred = predict_category(str(row['name']))
        if pred['category'] != 'Other':
            fixes.append({'id': row['id'], 'name': row['name'],
                          'old_cat': row['category'], 'new_cat': pred['category'],
                          'confidence': pred['confidence']})

    print(f'\n🏷️  Suggested fixes ({len(fixes)} transactions):')
    for fix in fixes[:15]:
        print(f'  {fix["name"]:<30} Other → {fix["new_cat"]:<16} ({fix["confidence"]}%)')

    if not dry_run and fixes:
        for fix in fixes:
            supabase.table('user_transactions').update(
                {'category': fix['new_cat']}
            ).eq('id', fix['id']).execute()
        print(f'\n✅ Updated {len(fixes)} transactions in Supabase!')
    elif dry_run:
        print(f'\nℹ️  Dry run — set dry_run=False to apply changes.')


fix_uncategorized_transactions(dry_run=True)

In [26]:
# ── Generate sample/synthetic data (for testing without real users) ───────────
def generate_synthetic_data(n_months=6, n_users=1):
    """
    Generate realistic fake transaction data for testing the ML pipeline
    when the DB is empty.
    """
    categories = {
        'Subscriptions': [('Spotify', 9.99), ('Netflix', 15.99), ('Disney+', 13.99)],
        'Dining':        [('Starbucks', 6.50), ('Uber Eats', 28.00), ('Restaurant', 45.00)],
        'Transport':     [('Uber', 18.00), ('Gas', 55.00), ('Parking', 12.00)],
        'Shopping':      [('Amazon', 35.00), ('Nike', 90.00), ('Target', 60.00)],
        'Bills':         [('Electricity', 95.00), ('Internet', 60.00), ('Phone', 45.00)],
        'Groceries':     [('Tesco', 75.00), ('Whole Foods', 120.00)],
    }

    rows = []
    now = datetime.now()

    for user_n in range(n_users):
        uid = f'demo-user-{user_n:04d}'
        balance = np.random.uniform(1000, 10000)

        for m in range(n_months):
            month_start = now - timedelta(days=30 * (n_months - m))

            # Deposit at start of month
            rows.append({
                'user_id': uid, 'name': 'Salary',
                'amount': round(np.random.uniform(2500, 5000), 2),
                'type': 'deposit', 'category': 'Income',
                'created_at': (month_start + timedelta(days=1)).isoformat()
            })

            # Random expenses throughout the month
            for cat, items in categories.items():
                n_txns = np.random.randint(1, 5)
                for _ in range(n_txns):
                    item_name, base_price = items[np.random.randint(len(items))]
                    amount = round(base_price * np.random.uniform(0.8, 1.4), 2)
                    day_offset = np.random.randint(0, 28)
                    rows.append({
                        'user_id': uid, 'name': item_name,
                        'amount': amount, 'type': 'expense', 'category': cat,
                        'created_at': (month_start + timedelta(days=day_offset)).isoformat()
                    })

    synthetic_df = pd.DataFrame(rows)
    synthetic_df['created_at'] = pd.to_datetime(synthetic_df['created_at'])
    synthetic_df['month'] = synthetic_df['created_at'].dt.to_period('M')
    synthetic_df['day_of_week'] = synthetic_df['created_at'].dt.day_name()
    return synthetic_df


# Use synthetic data if real DB is empty
if df.empty:
    print('📦 Generating synthetic data for testing...')
    df = generate_synthetic_data(n_months=6)
    expenses = df[df['type'] == 'expense']
    print(f'✅ Generated {len(df)} synthetic transactions ({len(expenses)} expenses)')
    print('   Re-run the analysis cells above to see results!')
else:
    print(f'✅ Using real data: {len(df)} transactions loaded from Supabase.')

📦 Generating synthetic data for testing...
✅ Generated 97 synthetic transactions (91 expenses)
   Re-run the analysis cells above to see results!


In [ ]:
 ── Export report to JSON ─────────────────────────────────────────────────────
if 'report' in dir() and isinstance(report, dict) and 'error' not in report:
    # Convert any non-serializable types
    def make_serializable(obj):
        if isinstance(obj, (pd.Period, pd.Timestamp)): return str(obj)
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if isinstance(obj, dict): return {k: make_serializable(v) for k, v in obj.items()}
        if isinstance(obj, list): return [make_serializable(i) for i in obj]
        return obj

    clean_report = make_serializable(report)

    with open('finscope_ai_report.json', 'w') as f:
        json.dump(clean_report, f, indent=2)

    print('✅ Report exported to finscope_ai_report.json')
    print(f'   Size: {len(json.dumps(clean_report)):,} characters')